In [ ]:
"""
DTL Dispatch Report — Interactive HU Analysis (Google Colab) — FIXED
==========================================================================
Fix: auto-detects the correct sheet and column names instead of assuming
"BHAVYA DATASET" -- so it works whether you upload dtl_.xlsx (raw export)
or DTL_Minimum_HU_Tool_with_data.xlsm (the tool file with data pasted in).
"""

# ---- CELL 1: Install/import what's needed ----
!pip install openpyxl networkx -q

import pandas as pd
import networkx as nx
from google.colab import files

# ---- CELL 2: Upload the file ----
print("Please choose your file to upload...")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"\nUploaded file: {file_name}")

# ---- CELL 3: Auto-detect the right sheet ----
xls = pd.ExcelFile(file_name)
print(f"Sheets found in this file: {xls.sheet_names}")

# Try to find a sheet that actually has the columns we need, instead of
# hardcoding a sheet name that may not exist in every file.
NEEDED_COLS_OPTIONS = [
    # (delivery_col, hu_col, load_status_col, hu_status_col)
    ("externalOrderId", "Handling_Unit", "Delivery_Loading_Status", "hu_status"),
    ("Delivery No.", "Handling_Unit", "Delivery_Loading_Status", "hu_status"),
]

df = None
delivery_col = hu_col = load_col = hustatus_col = None

for sheet in xls.sheet_names:
    temp = pd.read_excel(file_name, sheet_name=sheet)
    for (d, h, l, s) in NEEDED_COLS_OPTIONS:
        if all(c in temp.columns for c in (d, h, l, s)):
            df = temp
            delivery_col, hu_col, load_col, hustatus_col = d, h, l, s
            print(f"Using sheet: '{sheet}'  (columns matched: {d}, {h}, {l}, {s})")
            break
    if df is not None:
        break

if df is None:
    print("\nCould not auto-detect the right sheet/columns.")
    print("Here's what each sheet actually contains, so you can tell me:")
    for sheet in xls.sheet_names:
        temp = pd.read_excel(file_name, sheet_name=sheet, nrows=1)
        print(f"  '{sheet}': columns = {list(temp.columns)}")
    raise SystemExit("Fix the NEEDED_COLS_OPTIONS list above to match your real column names, then rerun.")

print(f"\nTotal rows loaded: {len(df)}")

# ---- CELL 4: Count all Partial / pending HUs (before closure) ----
loading_partial = df[df[load_col] == "Partial Completed"]
partial_hu_count = loading_partial[hu_col].nunique()

print("\n" + "=" * 55)
print(f"PARTIAL HUs FOUND (Loading = Partial Completed): {partial_hu_count}")
print("=" * 55)

# ---- CELL 5: Run the closure algorithm to get the MINIMUM HU set ----
pending_all = df[df[hustatus_col] != "LOADED"].dropna(subset=[delivery_col, hu_col])

G = nx.Graph()
for _, row in pending_all.iterrows():
    G.add_edge(("D", row[delivery_col]), ("H", row[hu_col]))

seed = loading_partial[loading_partial[hustatus_col] != "LOADED"]
seed_deliveries = set(seed[delivery_col].dropna().unique())

closure_hus, closure_deliveries, seen = set(), set(), set()
for d in seed_deliveries:
    node = ("D", d)
    if node not in G:
        continue
    comp = frozenset(nx.node_connected_component(G, node))
    if comp in seen:
        continue
    seen.add(comp)
    for n in comp:
        (closure_hus if n[0] == "H" else closure_deliveries).add(n[1])

# ---- CELL 6: Display the final answer ----
print("\n" + "=" * 55)
print(f"MINIMUM HUs REQUIRED (final answer): {len(closure_hus)}")
print("=" * 55)

# ---- CELL 7: Save and download the HU list ----
result_df = pd.DataFrame({hu_col: sorted(closure_hus)})
result_df.to_csv("minimum_hu_list.csv", index=False)
files.download("minimum_hu_list.csv")
print("\nDownloading minimum_hu_list.csv with all HU numbers...")

Please choose your file to upload...


Saving DTL_Minimum_HU_Tool_with_data.xlsm to DTL_Minimum_HU_Tool_with_data (1).xlsm

Uploaded file: DTL_Minimum_HU_Tool_with_data (1).xlsm
Sheets found in this file: ['InputData', 'Instructions', 'VBA_Code']
Using sheet: 'InputData'  (columns matched: Delivery No., Handling_Unit, Delivery_Loading_Status, hu_status)

Total rows loaded: 1851

PARTIAL HUs FOUND (Loading = Partial Completed): 99

MINIMUM HUs REQUIRED (final answer): 37


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>